# Protein sequence design with ProteinMPNN and LigandMPNN

# Switch to python 3.11 - run if using google colab
We need to use python 3.11 for LigandMPNN, not the more recent versions that are used by default by google colab

In [ ]:
%pip install -q condacolab

import condacolab
condacolab.install(python_version="3.11")

# Install packages and download github repos - only run if you do not have them installed

In [ ]:
%pip install numpy==1.23.5
%pip install torch ml-collections
%pip install -U ProDy
!git clone https://github.com/dauparas/ProteinMPNN.git
!git clone https://github.com/dauparas/LigandMPNN.git

# ProteinMPNN - full sequence design
Download a PDB file without a ligand to test Protein MPNN

In [ ]:
#pdb_path = specifies whihc PDB to run on, choose one from the PDB and try it or use example provided
#pdb_path_chains = which chain(s) to design
#out_folder = where to output results
#sampling_temp = higher temperature will sample amino acids with lower probability
#model_name = version of model to use
#seed = random seed

!python ProteinMPNN/protein_mpnn_run.py \
--pdb_path 1L2Y.pdb \
--pdb_path_chains A \
--out_folder outputs/test_run \
--num_seq_per_target 5 \
--sampling_temp 0.1 \
--model_name v_48_020 \
--seed 42


# Protein MPNN  - partial sequence design
Protein MPNN has helper scripts that will let us prepare inputs where only certain positions are allowe to vary

In [ ]:
# this script converts PDB(s) to a json file
!python ProteinMPNN/helper_scripts/parse_multiple_chains.py \
    --input_path 1L2Y.pdb \
    --output_path parsed_pdbs.jsonl

#script that edits the json file to specify fixed positions
#remove --specify_non_fixed to make the specified positions fixed instead of varied
!python ProteinMPNN/helper_scripts/make_fixed_positions_dict.py \
    --input_path parsed_pdbs.jsonl \
    --output_path fixed_positions.jsonl \
    --chain_list "A" \
    --position_list "6 18" \
    --specify_non_fixed

#run predictions with json to specify pdb and fixed positions
!python ProteinMPNN/protein_mpnn_run.py \
    --jsonl_path parsed_pdbs.jsonl \
    --fixed_positions_jsonl fixed_positions.jsonl \
    --out_folder outputs/fixed \
    --sampling_temp "0.1" \
    --num_seq_per_target 5

# LigandMPNN  - full sequence design
Note that LigandMPNN also comes with a ProteinMPNN model, you can change model type to "protein_mpnn" to run ProteinMPNN through LigandMPNN

In [ ]:
#model_type - ligand or protein mpnn
#chains_to_design - which chain(s) to design
#seed - random seed
#pdb_path - input pdb
#num_seq_per_target - how many sequences to output
#temperature - higher temperature will sample amino acids with lower probability
#out_folder - where output files will be written

!python LigandMPNN/run.py \
--model_type ligand_mpnn \
--chains_to_design A \
--seed 42 \
--pdb_path 10EI.pdb \
--num_seq_per_target 5 \
--temperature 0.1 \
--out_folder ./outputs/test_ligandmpnn

# LigandMPNN - redesign only some residues
We will redesign a loop near the ligand binding pocket

In [ ]:
#model_type - ligand or protein mpnn
#chains_to_design - which chain(s) to design
#seed - random seed
#pdb_path - input pdb
#num_seq_per_target - how many sequences to output
#temperature - higher temperature will sample amino acids with lower probability
#out_folder - where output files will be written

!python LigandMPNN/run.py \
--model_type ligand_mpnn \
--chains_to_design A \
--seed 42 \
--pdb_path 10EI.pdb \
--num_seq_per_target 5 \
--temperature 0.1 \
--redesigned_residues "A60 A61 A62 A63 A64 A65 A66 A67 A68 A69 A70" \
--out_folder ./outputs/test_ligandmpnn
